In [2]:
import sys
import time
import threading

from telemetrix import telemetrix

In [3]:
# board = telemetrix.Telemetrix(com_port="COM4", baud_rate=9600, start_loop=True)
board = telemetrix.Telemetrix()

LED_PIN = 4
BUTTON_PIN = 6
INTERRUPT_PIN = 3


board.set_pin_mode_digital_output(LED_PIN)
board.set_pin_mode_digital_input(BUTTON_PIN)



Telemetrix:  Version 1.47

Copyright (c) 2021-2025 Alan Yorinks All Rights Reserved.

Opening all potential serial ports...
	COM4

Waiting 4 seconds(arduino_wait) for Arduino devices to reset...
Valid Arduino ID Found.
Arduino compatible device found and connected to COM4
Waiting for Arduino to reset
Reset Complete

Retrieving Telemetrix4Arduino firmware ID...
Telemetrix4Arduino firmware version: 5.4.4


In [ ]:
# from documentation

# Callback data indices
CB_PIN_MODE = 0
CB_PIN = 1
CB_VALUE = 2
CB_TIME = 3


def the_callback(data):
    """
    A callback function to report data changes.
    This will print the pin number, its reported value and
    the date and time when the change occurred

    :param data: [pin, current reported value, pin_mode, timestamp]
    """
    date = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(data[CB_TIME]))
    print(f'Pin Mode: {data[CB_PIN_MODE]} Pin: {data[CB_PIN]} Value: {data[CB_VALUE]} Time Stamp: {date}')


def digital_in(my_board, pin):
    """
     This function establishes the pin as a
     digital input. Any changes on this pin will
     be reported through the call back function.

     :param my_board: a telemetrix instance
     :param pin: Arduino pin number
     """
    # set the pin mode
    my_board.set_pin_mode_digital_input(pin, the_callback)
    # time.sleep(1)
    # my_board.disable_all_reporting()
    # time.sleep(4)
    # my_board.enable_digital_reporting(12)
    # time.sleep(3)
    my_board.enable_digital_reporting(pin)
    # time.sleep(1)
    # print('Enter Control-C to quit.')
    # my_board.enable_digital_reporting(12)
    try:
        while True:
            time.sleep(.0001)
    except KeyboardInterrupt:
        board.shutdown()
        sys.exit(0)

def digital_read(my_board, pin):
    return my_board.set_pin_mode_digital_input(pin, return_value)
    try:
        while True:
            time.sleep(.0001)
    except KeyboardInterrupt:
        my_board.shutdown()
        sys.exit(0)


In [ ]:
def return_value(data):
    """
    A function to return the value of the pin when it changes.
    """
    pin = data[CB_PIN]
    value = data[CB_VALUE]
    if pin != BUTTON_PIN:
        return

board.digital_write(LED_PIN, 1 if value == 1 else 0)

board.set_pin_mode_digital_input(BUTTON_PIN, return_value)
board.enable_digital_reporting(BUTTON_PIN)

try:
    while True:
        time.sleep(.0001)
except KeyboardInterrupt:
    board.shutdown()
    sys.exit(0)

In [ ]:
try:
    while True:
        # button_state = board.digital_read(BUTTON_PIN)
        # button_state = digital_read(board, BUTTON_PIN)
        # print(button_state)
        digital_in(board, BUTTON_PIN)
        if button_state == 1:
            board.digital_write(LED_PIN, 1)  # Turn on the LED
        else:
            board.digital_write(LED_PIN, 0)  # Turn off the LED

        time.sleep(0.1)  # Add a small delay to avoid excessive CPU usage
except KeyboardInterrupt:
    board.shutdowm()
    sys.exit(0)

In [ ]:
import sys
import time
import threading

from telemetrix import telemetrix

board = telemetrix.Telemetrix()

LED_PIN = 4
BUTTON_PIN = 6

board.set_pin_mode_digital_output(LED_PIN)
board.set_pin_mode_digital_input(BUTTON_PIN)

# Callback data indices for telemetrix digital reporting
CB_PIN_MODE = 0
CB_PIN = 1
CB_VALUE = 2
CB_TIME = 3

def led_off(pin):
    board.digital_write(pin, 0)  # Turn off the LED
    print("LED off after 5 seconds")

timer = threading.Timer(5.0, led_off, args=[LED_PIN])

def button_callback(data):
    """Handle button state changes and update the LED."""
    pin = data[CB_PIN]
    value = data[CB_VALUE]
    # state = True
    if pin != BUTTON_PIN:
        return

    if value == 1:
        board.digital_write(LED_PIN, 1)
        print("Button pressed: LED on")
        timer.start()
    else:
        board.digital_write(LED_PIN, 0)
        print("Button released: LED off")


board.set_pin_mode_digital_output(LED_PIN)
board.set_pin_mode_digital_input(BUTTON_PIN, button_callback)
board.enable_digital_reporting(BUTTON_PIN)

try:
    while True:
        time.sleep(0.1)
except:
    board.shutdown()
    sys.exit(0)


Telemetrix:  Version 1.47

Copyright (c) 2021-2025 Alan Yorinks All Rights Reserved.

Opening all potential serial ports...
	COM4

Waiting 4 seconds(arduino_wait) for Arduino devices to reset...
Valid Arduino ID Found.
Arduino compatible device found and connected to COM4
Waiting for Arduino to reset
Reset Complete

Retrieving Telemetrix4Arduino firmware ID...
Telemetrix4Arduino firmware version: 5.4.4
Button pressed: LED on
LED off after 5 seconds
Button released: LED off


In [ ]:
# Imports
import sys
import time
import threading
import FreeSimpleGUI as sg
import serial

from telemetrix import telemetrix


# set up the GUI window
sg.theme('DefaultNoMoreNagging')

valueRead = 0

layout = [
    [sg.Text('Welcome to the firmata GUI exercise!')],
    [sg.Text('Enter the on time for the LED (in milliseconds)'), 
    sg.Input(key = '-INPUT-')],
    [sg.Text(key = '-OUTPUT1-')],
    [sg.Text(key = '-OUTPUT2-')],
    [sg.Button('OK'), sg.Button('Cancel')]
]

window = sg.Window('Firmata GUI', layout)


# set up the firmata
board = telemetrix.Telemetrix()

LED_PIN = 4
BUTTON_PIN = 6

board.set_pin_mode_digital_output(LED_PIN)
board.set_pin_mode_digital_input(BUTTON_PIN)

# Callback data indices for telemetrix digital reporting
CB_PIN_MODE = 0
CB_PIN = 1
CB_VALUE = 2
CB_TIME = 3

def led_off(pin):
    board.digital_write(pin, 0)  # Turn off the LED
    return "LED off"

timer = threading.Timer(valueRead, led_off, args=[LED_PIN])

def button_callback(data):
    """Handle button state changes and update the LED."""
    pin = data[CB_PIN]
    value = data[CB_VALUE]
    # state = True
    if pin != BUTTON_PIN:
        return

    state = True if value == 1 else False
    
    if state:
        # turn on the LED for the specified duration
        board.digital_write(LED_PIN, 1)
        print("Button pressed: LED on")
        timer.start()
    else:
        board.digital_write(LED_PIN, 0)
        print("Button released")


board.set_pin_mode_digital_output(LED_PIN)
board.set_pin_mode_digital_input(BUTTON_PIN, button_callback)
board.enable_digital_reporting(BUTTON_PIN)


# main loop
try:
    while True:
        time.sleep(0.1)
    #     event, values = window.read()
    #     if event == sg.WIN_CLOSED or event == 'Cancel':
    #         break
    #     if values['-INPUT-']:
    #         valueRead = int(values['-INPUT-'])
    #     window['-OUTPUT1-'].update("The LED is now configured to stay ON for " 
    #     + values['-INPUT-'] + " milliseconds")
    #     window['-OUTPUT2-'].update("Go ahead and press the button!")

    # window.close()

except KeyboardInterrupt:
    board.shutdown()
    sys.exit(0)
